In [1]:
SEED = 42

In [2]:
from pathlib import Path
from helpers.data.ticker_loader import load_tickers

TICKERS_FILE = Path("tickers.txt")
if not TICKERS_FILE.exists():
    TICKERS_FILE = Path("NN_Trading_project/tickers.txt")

# None        → all tickers
# ["AAPL", …] → explicit list
# 42          → random sample of 42 (seeded by SEED)
TICKER_SUBSET = None

TICKERS = load_tickers(TICKERS_FILE, subset=TICKER_SUBSET, seed=SEED)
print(f"Using {len(TICKERS)} tickers")

Using 61 tickers


In [ ]:
import pandas as pd
import datetime

INTERVAL   = "1d"
START_DATE = pd.Timestamp("2025-01-01")
END_DATE   = pd.Timestamp(datetime.date.today())

# Train: Date <= TRAIN_END_DATE
# Val:   (TRAIN_END_DATE, VAL_END_DATE]
# Test:  Date > VAL_END_DATE
TRAIN_END_DATE = pd.Timestamp("2025-11-30")
VAL_END_DATE   = pd.Timestamp("2026-01-20")

REBUILD_FEATURE_CACHE = True

In [ ]:
# Trading / Labeling
HORIZON_BARS     = 5
PROFIT_THRESHOLD = 0.15 / 100
STOP_LOSS        = -0.15 / 100
WINDOW           = 20

# Training
MAX_EPOCHS    = 50
PATIENCE      = 10
BUY_THRESHOLD = 0.5

# Optimizer / Model
BATCH_SIZE   = 64
HIDDEN_SIZES = [64]

# DataLoader
SPLIT_FRAC  = 0.85
NUM_WORKERS = 16

In [5]:
from helpers.data.date_config_manager import check_and_refresh_date_config

check_and_refresh_date_config(
    current_config={
        "INTERVAL":           str(INTERVAL),
        "START_DATE":         str(START_DATE.date()),
        "END_DATE":           str(END_DATE.date()),
        "TRAIN_END_DATE":     str(TRAIN_END_DATE.date()),
        "VAL_END_DATE":       str(VAL_END_DATE.date()),
        "TICKER_SUBSET":      str(TICKER_SUBSET),
    },
    config_path=Path.cwd() / "date_config.txt",
    stocks_dir=Path.cwd() / "dataset" / "stocks",
)

Date config unchanged — using cached CSVs where available.
  Config file: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/date_config.txt


False

In [6]:
from helpers.data.data_downloader import download_tickers

data_root  = Path.cwd() / "dataset"
stocks_dir = data_root / "stocks"

_summary = download_tickers(
    tickers=TICKERS, start=START_DATE, end=END_DATE,
    interval=INTERVAL, out_dir=stocks_dir,
)

 SKIP :: AGCO (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/AGCO.csv)
 SKIP :: ALLE (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/ALLE.csv)
 SKIP :: ALNY (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/ALNY.csv)
 SKIP :: AVB (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/AVB.csv)
 SKIP :: BR (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/BR.csv)
 SKIP :: BURL (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/BURL.csv)
 SKIP :: BWXT (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/BWXT.csv)
 SKIP :: CALM (cached CSV found at /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks/CALM.csv)
 SKIP :: CBOE (cached CSV found at /scratch/temp_/NN_Trading_Bot_Priva

# Building Features + Labels

In [7]:
import pickle
import shutil
import time

import torch
from torch.utils.data import DataLoader

from helpers.feature.feature_builder import precompute_and_cache, FEATURE_COLS
from helpers.data.dataset import StockDatasetSafe, is_cache_valid

root       = Path.cwd() / "dataset"
stocks_dir = root / "stocks"
assert stocks_dir.exists(), f"Missing: {stocks_dir}"

files       = sorted(stocks_dir.glob("*.csv"))
cache_dir   = Path.cwd() / f".feature_cache_forward_return_w{WINDOW}"
cache_dir.mkdir(parents=True, exist_ok=True)
scaler_path = cache_dir / "scaler.pkl"
index_path  = cache_dir / "index.pkl"

if REBUILD_FEATURE_CACHE:
    for p in [pp for pp in Path.cwd().glob(".feature*") if pp.exists()]:
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
        print(f"[Cache] Removed: {p}")
    time.sleep(1)

cache_ready = is_cache_valid(scaler_path, index_path)
if not cache_ready and cache_dir.exists():
    print("[Cache] Stale / incomplete — wiping cache dir.")
    shutil.rmtree(cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_FEATURE_CACHE or not cache_ready:
    scaler, index = precompute_and_cache(
        files=files, window=WINDOW, cache_dir=cache_dir,
        scaler_path=scaler_path, index_path=index_path,
        horizon_bars=HORIZON_BARS, train_end_date=TRAIN_END_DATE,
        val_end_date=VAL_END_DATE, profit_threshold=PROFIT_THRESHOLD,
        stop_loss=STOP_LOSS,
    )
else:
    print("[Cache] Using existing feature cache.")
    with open(scaler_path, "rb") as f: scaler = pickle.load(f)
    with open(index_path,  "rb") as f: index  = pickle.load(f)

train_ds = StockDatasetSafe(index, scaler, "train")
val_ds   = StockDatasetSafe(index, scaler, "val")
test_ds  = StockDatasetSafe(index, scaler, "test")

_pin    = torch.cuda.is_available()
assert _pin, "GPU required but torch.cuda.is_available() is False."
_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    pin_memory=_pin, persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)
train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kwargs)

xb, yb = next(iter(train_loader))
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
print(f"X batch: {xb.shape} {xb.dtype}  y batch: {yb.shape} {yb.dtype}")
print(f"Features: {len(FEATURE_COLS)} base × {WINDOW} lags = {len(FEATURE_COLS) * WINDOW}")

[Cache] Removed: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/.feature_cache_forward_return_w20
[Cache] Precomputing features (leakage-safe splits)...
[Cache] (1/61) AGCO.csv
[Cache] (2/61) ALLE.csv
[Cache] (3/61) ALNY.csv
[Cache] (4/61) AVB.csv
[Cache] (5/61) BR.csv
[Cache] (6/61) BURL.csv
[Cache] (7/61) BWXT.csv
[Cache] (8/61) CALM.csv
[Cache] (9/61) CBOE.csv
[Cache] (10/61) CMI.csv
[Cache] (11/61) CYBR.csv
[Cache] (12/61) DGX.csv
[Cache] (13/61) DOV.csv
[Cache] (14/61) EWBC.csv
[Cache] (15/61) EXR.csv
[Cache] (16/61) FAF.csv
[Cache] (17/61) FVRR.csv
[Cache] (18/61) GGG.csv
[Cache] (19/61) HIW.csv
[Cache] (20/61) HOPE.csv
[Cache] (21/61) HXL.csv
[Cache] (22/61) IAC.csv
[Cache] (23/61) IT.csv
[Cache] (24/61) ITW.csv
[Cache] (25/61) JBHT.csv
[Cache] (26/61) KLAC.csv
[Cache] (27/61) KMT.csv
[Cache] (28/61) LAZ.csv
[Cache] (29/61) LOGI.csv
[Cache] (30/61) LPLA.csv
[Cache] (31/61) MAA.csv
[Cache] (32/61) MCK.csv
[Cache] (33/61) MCO.csv
[Cache] (34/61) MLI.csv
[Cache] (35/61) M

# XGBoost

In [8]:
import numpy as np
import torch
import xgboost as xgb
import joblib
from sklearn.metrics import log_loss

from helpers.evaluation import buy_metrics, predict_probs_booster


def loader_to_numpy(loader):
    Xs, ys = [], []
    for xb, yb in loader:
        Xs.append(xb.numpy())
        ys.append(yb.numpy())
    return np.concatenate(Xs), np.concatenate(ys)


X_train, y_train = loader_to_numpy(train_loader)
X_val,   y_val   = loader_to_numpy(val_loader)
X_test,  y_test  = loader_to_numpy(test_loader)

num_pos = float((y_train == 1).sum())
num_neg = float((y_train == 0).sum())
scale_pos_weight = num_neg / max(1.0, num_pos)

print(f"Train {X_train.shape}  pos={int(num_pos)} neg={int(num_neg)}")
print(f"Val   {X_val.shape}    pos={int((y_val==1).sum())} neg={int((y_val==0).sum())}")
print(f"Test  {X_test.shape}   pos={int((y_test==1).sum())} neg={int((y_test==0).sum())}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

GPU_REQUIRED_MSG = "GPU required but torch.cuda.is_available() is False."
GPU_XGB_UNAVAILABLE_MSG = (
    "GPU required but XGBoost CUDA training is unavailable. "
    "Install GPU-enabled XGBoost/CUDA or run on a GPU machine."
)

GPU_XGB_PARAMS = {
    "tree_method": "hist",
    "device": "cuda",
}


def assert_cuda_available() -> None:
    assert torch.cuda.is_available(), GPU_REQUIRED_MSG


def assert_xgb_cuda_training_available(X: np.ndarray, y: np.ndarray) -> None:
    smoke_rows = min(2048, X.shape[0])
    dtrain_smoke = xgb.DMatrix(X[:smoke_rows], label=y[:smoke_rows])
    try:
        _ = xgb.train(
            params={
                "objective": "binary:logistic",
                "eval_metric": "logloss",
                "max_depth": 1,
                "eta": 0.3,
                **GPU_XGB_PARAMS,
            },
            dtrain=dtrain_smoke,
            num_boost_round=1,
            verbose_eval=False,
        )
    except Exception as e:
        raise AssertionError(GPU_XGB_UNAVAILABLE_MSG) from e


assert_cuda_available()
assert_xgb_cuda_training_available(X_train, y_train)

NUM_BOOST_ROUND       = 10000
EARLY_STOPPING_ROUNDS = PATIENCE

params = {
    "max_depth": 8, "eta": 0.001039, "subsample": 0.6544,
    "colsample_bytree": 0.4103, "min_child_weight": 20, "gamma": 2.092,
    "alpha": 0.1705, "lambda": 0.6038,
    "scale_pos_weight": scale_pos_weight,
    "objective": "binary:logistic", "eval_metric": "logloss",
    **GPU_XGB_PARAMS,
}

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

evals_result = {}
print(f"Training: max_rounds={NUM_BOOST_ROUND}, early_stop={EARLY_STOPPING_ROUNDS}")
booster = xgb.train(
    params=params, dtrain=dtrain, num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    evals_result=evals_result, verbose_eval=100,
)

best_ntree = int(booster.best_iteration + 1) if booster.best_iteration is not None else NUM_BOOST_ROUND
print(f"\nBest iteration: {best_ntree}")

probs_train = predict_probs_booster(booster, X_train, best_ntree)
probs_val   = predict_probs_booster(booster, X_val,   best_ntree)
probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
tr = buy_metrics(y_train, probs_train, BUY_THRESHOLD)
vl = buy_metrics(y_val,   probs_val,   BUY_THRESHOLD)
te = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)

print(f"Train  logloss={log_loss(y_train,probs_train):.6f}  acc={tr['acc']:.2f}%  P(success|BUY)={tr['buy_success']:.2f}%")
print(f"Val    logloss={log_loss(y_val,  probs_val  ):.6f}  acc={vl['acc']:.2f}%  P(success|BUY)={vl['buy_success']:.2f}%")
print(f"Test   logloss={log_loss(y_test, probs_test ):.6f}  acc={te['acc']:.2f}%  P(success|BUY)={te['buy_success']:.2f}%")

joblib.dump({"booster": booster, "best_ntree": best_ntree}, "best_model_xgb.pkl")
print("Saved → best_model_xgb.pkl")

Train (91129, 880)  pos=27353 neg=63776
Val   (13847, 880)    pos=4183 neg=9664
Test  (21952, 880)   pos=5613 neg=16339
scale_pos_weight = 2.3316
Training: max_rounds=10000, early_stop=10
[0]	train-logloss:0.69307	val-logloss:0.69313
[100]	train-logloss:0.68607	val-logloss:0.69122
[200]	train-logloss:0.67994	val-logloss:0.68978
[300]	train-logloss:0.67447	val-logloss:0.68874
[400]	train-logloss:0.66961	val-logloss:0.68795
[500]	train-logloss:0.66519	val-logloss:0.68741
[600]	train-logloss:0.66117	val-logloss:0.68702
[700]	train-logloss:0.65747	val-logloss:0.68671
[800]	train-logloss:0.65409	val-logloss:0.68651
[893]	train-logloss:0.65113	val-logloss:0.68641

Best iteration: 884
Train  logloss=0.651425  acc=61.95%  P(success|BUY)=42.49%
Val    logloss=0.686400  acc=50.65%  P(success|BUY)=34.74%
Test   logloss=0.652928  acc=62.92%  P(success|BUY)=33.67%
Saved → best_model_xgb.pkl


### Evaluate on Test Data

In [9]:
from helpers.evaluation import evaluate_on_test_data

df_test_preds, df_test_summary = evaluate_on_test_data(
    booster=booster, ntree=best_ntree,
    X_test=X_test, y_test=y_test,
    threshold=BUY_THRESHOLD, index_path=index_path,
    stocks_dir=stocks_dir, save_csv="test_predictions_full.csv",
)

Saved test predictions -> test_predictions_full.csv
P(success | BUY): 33.67%  |  acc: 62.92%


# Optuna Hyperparameter Search (XGBoost)

In [ ]:
import os
import numpy as np
import torch
import optuna
import wandb

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Keep notebook output compact during Optuna runs
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_CONSOLE"] = "off"
wandb_settings = wandb.Settings(silent=True, quiet=True, console="off")

wandb.login(key="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")

OPTUNA_N_TRIALS   = 50
OPTUNA_EARLY_STOP = max(10, PATIENCE * 5)
OPTUNA_MAX_ROUNDS = 3000

WANDB_PROJECT = "NN-Trading-Bot"
WANDB_GROUP   = f"optuna_xgb_{SEED}"

dtrain_opt = xgb.DMatrix(X_train, label=y_train)
dval_opt   = xgb.DMatrix(X_val,   label=y_val)
dtest_opt  = xgb.DMatrix(X_test,  label=y_test)

if "GPU_REQUIRED_MSG" not in globals():
    GPU_REQUIRED_MSG = "GPU required but torch.cuda.is_available() is False."
if "GPU_XGB_UNAVAILABLE_MSG" not in globals():
    GPU_XGB_UNAVAILABLE_MSG = (
        "GPU required but XGBoost CUDA training is unavailable. "
        "Install GPU-enabled XGBoost/CUDA or run on a GPU machine."
    )
if "GPU_XGB_PARAMS" not in globals():
    GPU_XGB_PARAMS = {
        "tree_method": "hist",
        "device": "cuda",
    }
if "assert_cuda_available" not in globals():
    def assert_cuda_available() -> None:
        assert torch.cuda.is_available(), GPU_REQUIRED_MSG
if "assert_xgb_cuda_training_available" not in globals():
    def assert_xgb_cuda_training_available(X: np.ndarray, y: np.ndarray) -> None:
        smoke_rows = min(2048, X.shape[0])
        dtrain_smoke = xgb.DMatrix(X[:smoke_rows], label=y[:smoke_rows])
        try:
            _ = xgb.train(
                params={
                    "objective": "binary:logistic",
                    "eval_metric": "logloss",
                    "max_depth": 1,
                    "eta": 0.3,
                    **GPU_XGB_PARAMS,
                },
                dtrain=dtrain_smoke,
                num_boost_round=1,
                verbose_eval=False,
            )
        except Exception as e:
            raise AssertionError(GPU_XGB_UNAVAILABLE_MSG) from e

assert_cuda_available()
assert_xgb_cuda_training_available(X_train, y_train)

TUNED_KEYS = ("max_depth", "eta", "subsample", "colsample_bytree",
              "min_child_weight", "gamma", "alpha", "lambda")


def _fmt_val(v):
    return f"{v:.4g}" if isinstance(v, float) else str(v)


def _make_run_name(d):
    return "____".join(f"{k}_{_fmt_val(d[k])}" for k in TUNED_KEYS)


def objective(trial: optuna.Trial) -> float:
    hp = {
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "eta":              trial.suggest_float("eta", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "alpha":            trial.suggest_float("alpha", 0.0, 10.0),
        "lambda":           trial.suggest_float("lambda", 0.5, 10.0),
    }
    params = {
        "objective": "binary:logistic", "eval_metric": "logloss",
        "seed": SEED,
        "scale_pos_weight": scale_pos_weight,
        **GPU_XGB_PARAMS,
        **hp,
    }

    run = wandb.init(
        project=WANDB_PROJECT, group=WANDB_GROUP,
        name=f"trial_{trial.number}__{_make_run_name(hp)}",
        config=hp,
        reinit="finish_previous",
        settings=wandb_settings,
    )

    evals_res = {}
    bst = xgb.train(
        params=params, dtrain=dtrain_opt, num_boost_round=OPTUNA_MAX_ROUNDS,
        evals=[(dtrain_opt, "train"), (dval_opt, "eval")],
        early_stopping_rounds=OPTUNA_EARLY_STOP,
        evals_result=evals_res, verbose_eval=False,
    )

    best_iter     = int(bst.best_iteration + 1) if bst.best_iteration is not None else OPTUNA_MAX_ROUNDS
    val_logloss   = evals_res["eval"]["logloss"][best_iter - 1]
    train_logloss = evals_res["train"]["logloss"][best_iter - 1]

    trial.set_user_attr("best_ntree",  best_iter)
    trial.set_user_attr("val_logloss", val_logloss)

    # Log per-round metrics — W&B natively renders train/eval logloss vs round
    for r, (tr_ll, ev_ll) in enumerate(zip(evals_res["train"]["logloss"], evals_res["eval"]["logloss"])):
        wandb.log({"train/logloss": tr_ll, "eval/logloss": ev_ll, "round": r + 1})

    wandb.summary["eval/best_ntree"] = best_iter
    wandb.summary["train/final_logloss"] = train_logloss
    wandb.summary["eval/final_logloss"] = val_logloss

    run.finish()

    # Clean per-trial print (current trial not yet recorded, so default to val_logloss)
    prev_best = min((t.value for t in trial.study.trials if t.value is not None), default=val_logloss)
    best_so_far = min(prev_best, val_logloss)
    marker = " *" if val_logloss <= best_so_far else ""
    print(f"  [{trial.number + 1:3d}/{OPTUNA_N_TRIALS}]  train={train_logloss:.6f}  val={val_logloss:.6f}  rounds={best_iter}{marker}")

    return val_logloss


study = optuna.create_study(
    direction="minimize", study_name="xgb_hparam_search",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

print(f"Starting Optuna search: {OPTUNA_N_TRIALS} trials …")
print(f"{'':>6}{'trial':>8}  {'train_loss':>12}  {'val_loss':>12}  {'rounds':>8}")
print(f"{'':>6}{'-'*8}  {'-'*12}  {'-'*12}  {'-'*8}")
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=False)

best_trial  = study.best_trial
best_params = best_trial.params
print(f"\nBest trial #{best_trial.number}  val_logloss={best_trial.value:.6f}")
print("Best params:", best_params)

# ── Final run: retrain with best params ──────────────────────────────────────

final_run = wandb.init(
    project=WANDB_PROJECT, group=WANDB_GROUP,
    name=f"BEST_trial_{best_trial.number}__{_make_run_name(best_params)}",
    config=best_params,
    reinit="finish_previous",
    settings=wandb_settings,
)

# ── Retrain on train+val with best params ────────────────────────────────────

final_params = {
    "objective": "binary:logistic", "eval_metric": "logloss",
    "seed": SEED,
    "scale_pos_weight": scale_pos_weight,
    **GPU_XGB_PARAMS,
    **best_params,
}
best_ntree_optuna = int(best_trial.user_attrs["best_ntree"])

dtrain_full = xgb.DMatrix(
    np.concatenate([X_train, X_val]),
    label=np.concatenate([y_train, y_val]),
)
print(f"\nRetraining on train+val for {best_ntree_optuna} rounds …")
booster = xgb.train(
    params=final_params, dtrain=dtrain_full,
    num_boost_round=best_ntree_optuna, verbose_eval=False,
)
best_ntree = best_ntree_optuna

probs_test_optuna  = predict_probs_booster(booster, X_test,  best_ntree)
probs_train_optuna = predict_probs_booster(booster, X_train, best_ntree)
te_opt = buy_metrics(y_test,  probs_test_optuna,  BUY_THRESHOLD)
tr_opt = buy_metrics(y_train, probs_train_optuna, BUY_THRESHOLD)

print(f"Train  acc={tr_opt['acc']:.2f}%  P(success|BUY)={tr_opt['buy_success']:.2f}%")
print(f"Test   acc={te_opt['acc']:.2f}%  P(success|BUY)={te_opt['buy_success']:.2f}%  logloss={log_loss(y_test, probs_test_optuna):.6f}")

wandb.log({
    "train/final_logloss":   log_loss(y_train, probs_train_optuna),
    "train/accuracy":        tr_opt["acc"],
    "train/buy_success":     tr_opt["buy_success"],
    "eval/test_logloss":     log_loss(y_test, probs_test_optuna),
    "eval/test_accuracy":    te_opt["acc"],
    "eval/test_buy_success": te_opt["buy_success"],
})

joblib.dump({"booster": booster, "best_ntree": best_ntree}, "best_model_xgb.pkl")
print("\nSaved → best_model_xgb.pkl")
print(f"Use BUY_THRESHOLD = {BUY_THRESHOLD}")

final_run.finish()
print("W&B runs finished.")

/scratch/temp_/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Optuna search: 50 trials …
         trial    train_loss      val_loss    rounds
      --------  ------------  ------------  --------
  [  1/50]  train=0.671624  val=0.689686  rounds=4 *
  [  2/50]  train=0.670856  val=0.688150  rounds=10 *
  [  3/50]  train=0.676262  val=0.689541  rounds=31
  [  4/50]  train=0.673391  val=0.687590  rounds=7 *
  [  5/50]  train=0.675041  val=0.689819  rounds=5
  [  6/50]  train=0.678088  val=0.689063  rounds=47
  [  7/50]  train=0.673215  val=0.689105  rounds=232
  [  8/50]  train=0.676446  val=0.689092  rounds=326
  [  9/50]  train=0.675298  val=0.688902  rounds=38
  [ 10/50]  train=0.679799  val=0.689019  rounds=6
  [ 11/50]  train=0.643145  val=0.686438  rounds=1270 *
  [ 12/50]  train=0.644465  val=0.686506  rounds=1216
  [ 13/50]  train=0.642518  val=0.686612  rounds=1276
  [ 14/50]  train=0.646652  val=0.686690  rounds=994
  [ 15/50]  train=0.662413  val=0.688018  rounds=208
  [ 16/50]  train=0.661659  val=0.688228  rounds=161
  [ 17/50] 

# Eval Data Analysis

In [ ]:
from helpers.evaluation import evaluate_split_from_artifacts

SELECTED_THRESHOLD = 0.9

val_preds, daily, val_summary = evaluate_split_from_artifacts(
    split="val", threshold=SELECTED_THRESHOLD,
    index_path=index_path, scaler_path=scaler_path,
    model_path="best_model_xgb.pkl", verbose=True,
)
display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "val_analysis/threshold":    SELECTED_THRESHOLD,
        "val_analysis/total_trades": int(val_summary["total_trades"]),
        "val_analysis/pct_success":  float(val_summary["pct_success"]),
        "val_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })

# Test Data Analysis

In [ ]:
SELECTED_THRESHOLD = 0.9

test_preds, daily, test_summary = evaluate_split_from_artifacts(
    split="test", threshold=SELECTED_THRESHOLD,
    index_path=index_path, scaler_path=scaler_path,
    model_path="best_model_xgb.pkl", verbose=True,
)
display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "test_analysis/threshold":    SELECTED_THRESHOLD,
        "test_analysis/total_trades": int(test_summary["total_trades"]),
        "test_analysis/pct_success":  float(test_summary["pct_success"]),
        "test_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })